In [1]:
# ================================
# LinkedIn Post Generator (BLUE THEME)
# Simple & Beautiful Design
# ================================

# Install packages
!pip install -q gradio groq

import os
import json
from datetime import datetime
import gradio as gr
from groq import Groq

# ========================
# API KEY (Colab Secrets)
# ========================
from google.colab import userdata
try:
    api_key = userdata.get("GROQ_API_KEY")
    if not api_key:
        raise ValueError("GROQ_API_KEY not found")
    os.environ["GROQ_API_KEY"] = api_key
    print("✅ API Key loaded successfully")
except Exception as e:
    print(f"❌ Error: {e}")
    print("Add GROQ_API_KEY to Colab Secrets (🔑 icon)")

client = Groq(api_key=os.environ["GROQ_API_KEY"])


# ========================
# FEATURE 1: Generate Post
# ========================
def generate_post(topic, tone, length, hashtags, emojis, cta):
    if not topic:
        return "❌ Please enter a topic"

    prompt = f"""Write a LinkedIn post.

Topic: {topic}
Tone: {tone}
Length: {length}
Hashtags: {'Yes' if hashtags else 'No'}
Emojis: {'Yes' if emojis else 'No'}
CTA: {cta if cta else 'None'}

Rules:
- Strong hook in first line
- Professional LinkedIn style
- Human and engaging
- Valuable content
"""

    try:
        response = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=800
        )
        return response.choices[0].message.content

    except Exception as e:
        return f"❌ Error: {str(e)}"


# ========================
# FEATURE 2: Word Counter
# ========================
def count_stats(text):
    if not text or "❌" in text:
        return "Generate a post first"

    words = len(text.split())
    chars = len(text)
    linkedin_limit = 3000

    if chars > linkedin_limit:
        status = f"⚠️ OVER by {chars - linkedin_limit} chars"
    else:
        remaining = linkedin_limit - chars
        status = f"✅ {remaining} chars left"

    return f"Words: {words} | Chars: {chars} | {status}"


# ========================
# FEATURE 3: Generate Multiple
# ========================
def generate_multiple(topic, tone, length, hashtags, emojis, cta):
    if not topic:
        return "❌ Please enter a topic"

    posts = []
    tones = ["Professional", "Casual", "Inspirational"]

    for i, custom_tone in enumerate(tones, 1):
        post = generate_post(topic, custom_tone, length, hashtags, emojis, cta)
        posts.append(f"Version {i} ({custom_tone}):\n{post}\n\n{'='*50}\n\n")

    return "".join(posts)


# ========================
# FEATURE 4: Improve Post
# ========================
def improve_post(post):
    if not post or "❌" in post:
        return "❌ Generate a post first"

    prompt = f"""Make this LinkedIn post better for engagement:

{post}

Return ONLY the improved post."""

    try:
        response = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=1000
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"❌ Error: {str(e)}"


# ========================
# FEATURE 5: Hashtags
# ========================
def suggest_hashtags(topic):
    if not topic:
        return "❌ Enter a topic first"

    prompt = f"""Give 10 LinkedIn hashtags for: {topic}
Format: #hashtag1 #hashtag2 etc."""

    try:
        response = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=200
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"❌ Error: {str(e)}"


# ========================
# FEATURE 6: Save Post
# ========================
def save_post(post, topic):
    if not post or "❌" in post:
        return "❌ Generate a post first"

    try:
        filename = f"post_{topic[:15].replace(' ', '_')}_{datetime.now().strftime('%Y%m%d_%H%M')}.txt"
        with open(filename, 'w', encoding='utf-8') as f:
            f.write(f"Topic: {topic}\n")
            f.write(f"Date: {datetime.now().strftime('%Y-%m-%d %H:%M')}\n")
            f.write("="*50 + "\n\n")
            f.write(post)
        return f"✅ Saved: {filename}"
    except Exception as e:
        return f"❌ Error: {str(e)}"


# ========================
# FEATURE 7: Analytics
# ========================
def analyze_post(post):
    if not post or "❌" in post:
        return "Generate a post first"

    words = len(post.split())
    chars = len(post)
    sentences = len([s for s in post.split('.') if s.strip()])
    paragraphs = len([p for p in post.split('\n\n') if p.strip()])

    if words > 0 and sentences > 0:
        avg = words / sentences
        if avg < 15:
            readability = "Excellent"
        elif avg < 20:
            readability = "Good"
        else:
            readability = "Could be simpler"
    else:
        readability = "N/A"

    return f"Words: {words} | Sentences: {sentences} | Paragraphs: {paragraphs} | Readability: {readability}"


# ========================
# FEATURE 8: Reading Time
# ========================
def get_reading_time(post):
    if not post or "❌" in post:
        return "Generate a post first"

    words = len(post.split())
    reading_seconds = (words / 200) * 60

    if reading_seconds < 30:
        return f"Reading time: {int(reading_seconds)} seconds"
    elif reading_seconds < 120:
        return f"Reading time: {int(reading_seconds // 60)} min {int(reading_seconds % 60)} sec"
    else:
        minutes = reading_seconds / 60
        return f"Reading time: {minutes:.1f} minutes"


# ========================
# CUSTOM CSS - BLUE THEME WITH BLACK BACKGROUND
# ========================
custom_css = """
/* Main Background - BLACK */
body {
    background: #000000;
}

.gradio-container {
    background: #1a1a1a;
    border-radius: 15px;
    box-shadow: 0 8px 32px rgba(42, 82, 152, 0.3);
    padding: 25px;
}

/* Headers */
h1 { color: #ffffff; font-weight: bold; font-size: 36px; }
h2 { color: #ffffff; font-weight: bold; }
h3 { color: #87ceeb; font-weight: 600; }

/* Input Boxes */
.gradio-textbox input,
.gradio-textbox textarea {
    border: 2px solid #2a5298;
    border-radius: 8px;
    padding: 12px;
    background: #2a2a2a;
    color: #ffffff;
}

.gradio-textbox input:focus,
.gradio-textbox textarea:focus {
    border-color: #1e3c72;
    box-shadow: 0 0 8px rgba(30, 60, 114, 0.3);
}

/* Dropdowns */
.gradio-dropdown select {
    border: 2px solid #2a5298;
    border-radius: 8px;
    padding: 10px;
    background-color: #2a2a2a;
    color: #ffffff;
}

/* Checkboxes */
.gradio-checkbox {
    accent-color: #2a5298;
}

/* Primary Button - Blue */
button[variant="primary"] {
    background: linear-gradient(135deg, #1e3c72 0%, #2a5298 100%);
    color: white;
    border: none;
    border-radius: 8px;
    font-weight: bold;
    box-shadow: 0 4px 12px rgba(42, 82, 152, 0.3);
    transition: all 0.3s ease;
}

button[variant="primary"]:hover {
    transform: translateY(-2px);
    box-shadow: 0 6px 16px rgba(42, 82, 152, 0.4);
}

/* Secondary Buttons - Light Blue */
button:not([variant="primary"]) {
    background: linear-gradient(135deg, #5ba3d0 0%, #2a5298 100%);
    color: white;
    border: none;
    border-radius: 8px;
    font-weight: bold;
    box-shadow: 0 4px 12px rgba(91, 163, 208, 0.3);
    transition: all 0.3s ease;
}

button:not([variant="primary"]):hover {
    transform: translateY(-2px);
    box-shadow: 0 6px 16px rgba(91, 163, 208, 0.4);
}

/* Output Boxes */
.gradio-textbox[interactive="false"] {
    background: #2a2a2a;
    border: 2px solid #2a5298;
    border-radius: 8px;
    color: #ffffff;
}

/* Labels */
label {
    color: #ffffff;
    font-weight: 600;
}

/* Groups */
.gradio-group {
    border: 2px solid #2a5298;
    border-radius: 10px;
    padding: 18px;
    background: #252525;
}

/* Separator */
hr {
    border: 2px solid #2a5298;
    opacity: 0.2;
}

/* Info text */
.info {
    color: #2a5298;
    font-size: 12px;
    margin-top: 5px;
}
"""

# ========================
# GRADIO UI
# ========================
with gr.Blocks(
    title="LinkedIn Post Generator",
    theme=gr.themes.Soft(),
    css=custom_css
) as demo:

    # Header
    gr.Markdown("""
    <div style="text-align: center; padding: 25px; background: linear-gradient(135deg, #1e3c72 0%, #2a5298 100%); border-radius: 12px; color: white; margin-bottom: 20px;">
        <h1 style="font-size: 40px; margin: 0; color: white;">🚀 LinkedIn Post Generator</h1>
        <p style="font-size: 16px; margin-top: 8px; opacity: 0.95;">Create awesome LinkedIn posts in seconds</p>
    </div>
    """)

    # Main Content
    with gr.Row():
        # Left - Inputs
        with gr.Column(scale=1):
            with gr.Group():
                gr.Markdown("### 📝 Write Your Topic")

                topic = gr.Textbox(
                    label="What's your post about?",
                    placeholder="Example: How I built a startup and raised $2M",
                    lines=3
                )

                cta = gr.Textbox(
                    label="What should people do? (optional)",
                    placeholder="Example: Share your experience in comments"
                )

        # Right - Settings
        with gr.Column(scale=1):
            with gr.Group():
                gr.Markdown("### ⚙️ Settings")

                tone = gr.Dropdown(
                    choices=["Professional", "Casual", "Inspirational", "Educational"],
                    value="Professional",
                    label="Choose tone"
                )

                length = gr.Dropdown(
                    choices=["Short", "Medium", "Long"],
                    value="Medium",
                    label="Post length"
                )

                hashtags = gr.Checkbox(
                    label="Add hashtags",
                    value=True
                )

                emojis = gr.Checkbox(
                    label="Add emojis",
                    value=True
                )

    # Main Buttons
    gr.Markdown("---")
    with gr.Row():
        generate_btn = gr.Button(
            "✨ Generate Post",
            variant="primary",
            size="lg"
        )
        multi_btn = gr.Button(
            "🔄 Try 3 Versions",
            size="lg"
        )
        hashtag_btn = gr.Button(
            "🏷️ Get Hashtags",
            size="lg"
        )

    # Output
    with gr.Group():
        gr.Markdown("### 📄 Your Post")
        output = gr.Textbox(
            label="Post content",
            lines=15,
            interactive=False,
            show_copy_button=True
        )

    # Analytics
    with gr.Group():
        gr.Markdown("### 📊 Post Info")
        with gr.Row():
            counter_output = gr.Textbox(
                label="Stats",
                interactive=False,
                lines=2
            )
            stats_output = gr.Textbox(
                label="Analysis",
                interactive=False,
                lines=2
            )
            reading_time_output = gr.Textbox(
                label="Reading Time",
                interactive=False,
                lines=2
            )

    # Action Buttons
    gr.Markdown("---")
    with gr.Row():
        improve_btn = gr.Button(
            "✨ Make It Better",
            size="lg"
        )
        save_btn = gr.Button(
            "💾 Save It",
            size="lg"
        )

    # Results
    with gr.Row():
        improve_output = gr.Textbox(
            label="Improved version",
            lines=12,
            interactive=False,
            show_copy_button=True
        )
        hashtag_output = gr.Textbox(
            label="Hashtags",
            lines=5,
            interactive=False
        )

    # Status
    with gr.Group():
        save_status = gr.Textbox(
            label="Status",
            interactive=False
        )

    # Tips
    with gr.Group():
        gr.Markdown("""
        **💡 Tips:**
        - Be specific about your topic
        - Share real experience or numbers
        - Ask questions to get engagement
        - Edit the post to add your voice
        - Post Tuesday-Thursday between 9-10 AM
        """)

    # ========================
    # Event Handlers
    # ========================

    generate_btn.click(
        fn=generate_post,
        inputs=[topic, tone, length, hashtags, emojis, cta],
        outputs=[output]
    )

    output.change(fn=count_stats, inputs=[output], outputs=[counter_output])
    output.change(fn=analyze_post, inputs=[output], outputs=[stats_output])
    output.change(fn=get_reading_time, inputs=[output], outputs=[reading_time_output])

    multi_btn.click(
        fn=generate_multiple,
        inputs=[topic, tone, length, hashtags, emojis, cta],
        outputs=[output]
    )

    hashtag_btn.click(
        fn=suggest_hashtags,
        inputs=[topic],
        outputs=[hashtag_output]
    )

    improve_btn.click(
        fn=improve_post,
        inputs=[output],
        outputs=[improve_output]
    )

    save_btn.click(
        fn=save_post,
        inputs=[output, topic],
        outputs=[save_status]
    )


# Launch
print("\n" + "="*50)
print("🚀 LinkedIn Post Generator Starting...")
print("="*50)
print("\n✨ Blue theme + Simple design")
print("\nFeatures:")
print("  ✅ Generate Posts")
print("  ✅ 3 Versions")
print("  ✅ Hashtags")
print("  ✅ Improve Posts")
print("  ✅ Save to File")
print("\n" + "="*50 + "\n")

demo.launch(share=True, debug=False)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 2.9 MB/s eta 0:00:00
✅ API Key loaded successfully


/tmp/ipykernel_578/868276119.py:335: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_578/868276119.py:335: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(



🚀 LinkedIn Post Generator Starting...

✨ Blue theme + Simple design

Features:
  ✅ Generate Posts
  ✅ 3 Versions
  ✅ Hashtags
  ✅ Improve Posts
  ✅ Save to File


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://54538e3cc0313c3a57.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
